In [1]:
#Library Imports
import folium.map
import folium.plugins.heat_map
import requests
import folium
import sqlite3
import pandas as pd
from math import radians, cos, sin, asin, sqrt
from datetime import datetime as dt
import random

In [2]:
#Globals
customer_dict = {} #storage of valid customer objects
distributioncentre_dict = {} #storage of valid distribution centre objects
distributioncentre_output_lst = []
heat_data = []
global_traffic_locations = []
route_lon = []
route_lat = []
total_activity = 0
total_active_lat = 0
total_active_lon = 0

class Customer():
    def __init__(self, id, housenum, postcode, status):
        """Initialises the customer object and sets the values of it's properties after it confirms they are valid."""
        if self.validate(id, postcode):
            self.id = str(id)
            self.housenum = housenum
            self.postcode = postcode
            self.status = status
            self.insert()
            self.find_distribution_centre(self.latitude, self.longitude)

    def validate(self, id, postcode):
        """Validates that the customer's id hasn't already been used and that the postcode is real."""
        if id in customer_dict:
            return False
       
       #Gets the data of the postcode from the Postcode.io service
        postcodeio_response = requests.get("https://api.postcodes.io/postcodes/"+postcode)
        if postcodeio_response.status_code != 200:
            return False
        else:
            self.latitude = postcodeio_response.json()["result"]["latitude"]
            self.longitude = postcodeio_response.json()["result"]["longitude"]
        
        return True
    
    def insert(self):
        """Inserts the customer object data in a dictionary."""
        customer_dict[self.id] = {"House Number":self.housenum, "Postcode":self.postcode, "Latitude":self.latitude, "Longitude":self.longitude, "Status":self.status}

    def add(self, key, value):
        """Adds a key and a value to the customer dictionary."""
        customer_dict[key] = value

    def find_distribution_centre(self, latitude, longitide):
        """Finds the closest distribution centre to the customer, using the haversine formula. Stores this in the customer dictionary."""
        self.closest_distributioncentre = ""
        self.distance_to_centre = 1000000000000000000000000000

        for distributioncentre in distributioncentre_dict:
            #Haversine Formula - distance between two points on a sphere
            distance = 3956 * (2 * asin(sqrt(sin((radians(float(distributioncentre_dict[distributioncentre]["Latitude"])) - radians(float(latitude))) / 2)**2 + cos(radians(float(latitude))) * cos(radians(float(distributioncentre_dict[distributioncentre]["Latitude"]))) * sin((radians(float(distributioncentre_dict[distributioncentre]["Longitude"])) - radians(float(longitide))) / 2)**2)))

            if distance < self.distance_to_centre:
                self.closest_distributioncentre = distributioncentre
                self.distance_to_centre = distance

            distributioncentre_dict[distributioncentre]["Customer Distances"].append({self.id:distance})

        customer_dict[str(self.id)]["Closest Distribution Centre"] = self.closest_distributioncentre
        customer_dict[str(self.id)]["Distance to Centre"] = self.distance_to_centre
        distributioncentre_dict[self.closest_distributioncentre]["Number of Customers"] += 1

class WebTraffic(Customer):
    def __init__(self, datetime, customerid):
        """Initialises the web traffic object and sets the values of it's properties after it confirms they are valid."""
        if self.validate(datetime, customerid):
            self.datetime = datetime
            self.customerid = str(customerid)
            self.insert()
            self.add_to_heatmap()

    def validate(self, datetime, customerid):
        """Validates that the web traffic is referring to a customer that exists, has a valid date time, and that the customer doesn't already have a web traffic record at that time."""
        if str(customerid) in customer_dict:

            if str(datetime) in customer_dict[customerid]:
                return False
            
            try:
                datetime = dt(int(datetime[6:10]), int(datetime[3:5]), int(datetime[0:2]), int(datetime[11:13]), int(datetime[14:16]))
            except:
                return False
                
            return True
        
        else:
            return False
        
    def insert(self):
        """Inserts the web traffic object into the customer dictionary, where the id's match."""
        customer_dict[self.customerid][str(self.datetime)] = {"Date":self.datetime}
        distributioncentre_dict[customer_dict[self.customerid]["Closest Distribution Centre"]]["Nearby Activity"] += 1

        global total_activity, total_active_lat, total_active_lon
        
        total_activity += 1
        global_traffic_locations.append((customer_dict[self.customerid]["Latitude"], customer_dict[self.customerid]["Longitude"]))
        total_active_lat += customer_dict[self.customerid]["Latitude"]
        total_active_lon += customer_dict[self.customerid]["Longitude"]

    def add_to_heatmap(self):
        """Adds the cooridinates of the customer that created the web traffic to the heat_data array. This is so it will appear on the folium map."""
        heat_data.append([customer_dict[self.customerid]["Latitude"], customer_dict[self.customerid]["Longitude"]])

class DistributionCentre():
    def __init__(self, name, address):
        """Initialises the distribution centre object and sets the values of it's properties after it confirms they are valid."""
        if self.validate(name, address):
            self.name = name
            self.address = address
            self.insert()
            self.addtomap()

    def validate(self, name, address):
        """Validates that the distribution centre name doesn't already exist and that the postcode is valid."""
        if name  in distributioncentre_dict:
            return False
        
        #Gets the data of the postcode from the Postcode.io service
        postcodeio_response = requests.get("https://api.postcodes.io/postcodes/"+address[len(address)-8:len(address)])
        if postcodeio_response.status_code == 200:
            self.latitude = postcodeio_response.json()["result"]["latitude"]
            self.longitude = postcodeio_response.json()["result"]["longitude"]
        else:
            return False
        
        return True


    def insert(self):
        """Inserts the distribution centre object into the distribution centre dictionary."""
        distributioncentre_dict[self.name] = {"address":self.address, "Latitude":self.latitude, "Longitude":self.longitude, "Customer Distances": [], "Number of Customers":0, "Nearby Activity":0}

    def addtomap(self):
        """Adds a marker to the folium map of where the distribution centre is."""
        folium.Marker(location=[self.latitude, self.longitude], popup=(self.name + " Distribution Centre")).add_to(m)


In [3]:
#Data import
cust_response = pd.read_csv("https://github.com/sampenrice/adtl5dba/blob/main/Customer_details.csv?raw=true")
traffic_response = pd.read_csv("https://github.com/sampenrice/adtl5dba/blob/main/Traffic.csv?raw=true")
dist_response = pd.read_csv("https://github.com/sampenrice/adtl5dba/blob/main/Distribution_centres.csv?raw=true")

In [4]:
#Create the map
m = folium.Map(location=[52,1],zoom_start=6)

#Iterate through the distribution centre csv and create a DistributionCentre object for each
for row in dist_response.itertuples():
    DistributionCentre(row[1], row[2])

#Iterate through the customer csv and create a Customer object for each
for row in cust_response.itertuples():
    Customer(row[1],row[2],row[3],row[4])

#Iterate through the web traffic csv and create a WebTraffic object for each
for row in traffic_response.itertuples():
    WebTraffic(row[1], row[2])

In [5]:
#Data enrichment
folium.plugins.HeatMap(heat_data).add_to(m) #Add the array of cooridantes of valid web traffic locations to the map

m #displays the map

In [6]:
#Displaying Stock Locations
def display_table():
    """Returns a string of the list of valid distribution centres and the number of customer of which they are closest to."""
    table = "{:<30} {:<15}".format('Distribution Centre','Number of Customers')
    for row in distributioncentre_dict:
        table += "\n{:<30} {:<15}".format(row, str(distributioncentre_dict[row]["Number of Customers"]))
    
    return table

def recommend_stock_distribution():
    """Returns a string, listing valid distribution centres and the % of stock to provide them with. Calculated by the proportion of nearby web traffic they have."""
    return_string = "{:<30} {:<15}".format('Distribution Centre','Recommended % Proportion of Stock')
    for centre in distributioncentre_dict:
        return_string += "\n{:<30} {:<15}".format(centre, str(round(distributioncentre_dict[centre]["Nearby Activity"] / total_activity * 100, 3)))

    return return_string

print(display_table())
print("\n"+recommend_stock_distribution())

Distribution Centre            Number of Customers
Bristol                        16             
London                         21             
Leeds                          10             
Northallerton                  21             
Cardiff                        0              
Norwich                        24             
Cambridge                      0              

Distribution Centre            Recommended % Proportion of Stock
Bristol                        8.5            
London                         23.459         
Leeds                          6.193          
Northallerton                  35.514         
Cardiff                        0.0            
Norwich                        26.334         
Cambridge                      0.0            


In [7]:
#Engineer Placement Algorithm
def recommend_engineer_placement():
    """Returns the best distribution centres for 2 engineers to be placed at, based on the locations of customers with web traffic records. 
    The more web traffic records, the more weighting to being closer to that customer."""   

    distance_to_centre1 = 100000000000000000000000000
    distance_to_centre2 = 100000000000000000000000000

    global total_activity, total_active_lat, total_active_lon 
    
    average_active_lat = total_active_lat / total_activity
    average_active_lon = total_active_lon / total_activity

    for distributioncentre in distributioncentre_dict:
        distance = 3956 * (2 * asin(sqrt(sin((radians(float(distributioncentre_dict[distributioncentre]["Latitude"])) - radians(float(average_active_lat))) / 2)**2 + cos(radians(float(average_active_lat))) * cos(radians(float(distributioncentre_dict[distributioncentre]["Latitude"]))) * sin((radians(float(distributioncentre_dict[distributioncentre]["Longitude"])) - radians(float(average_active_lon))) / 2)**2)))

        if distance < distance_to_centre1:
            first_closest_distributioncentre = distributioncentre
            distance_to_centre1 = distance
        
        elif distance < distance_to_centre2:
            second_closest_distributioncentre = distributioncentre
            distance_to_centre2 = distance

    return ("Best Engineer Positions: " + second_closest_distributioncentre + ", " + first_closest_distributioncentre)  

print("\n"+str(recommend_engineer_placement()))


Best Engineer Positions: Norwich, Cambridge


In [ ]:
#Route Mapping
key = input("Enter TomTom API Key: ")

def route_to_customer(customer):
    """Plots a route from a given customer to it's nearest distribution centre."""
    customer_postcodeio_response = requests.get("https://api.postcodes.io/postcodes/"+customer["Postcode"])
    customer_response_dict = customer_postcodeio_response.json()

    if customer_postcodeio_response.status_code != 200:
         raise Exception("Random has chosen a customer with an invalid postcode (according to postcode.io). Please try run this again and see if random chooses a suitable test.")

    #Gets the route data from the TomTom api service
    response = requests.get("https://api.tomtom.com/routing/1/calculateRoute/"+str(customer_response_dict["result"]["latitude"])+","+str(customer_response_dict["result"]["longitude"])+":"+str(distributioncentre_dict[customer["Closest Distribution Centre"]]["Latitude"])+","+str(distributioncentre_dict[customer["Closest Distribution Centre"]]["Longitude"])+"/json?traffic=true&avoid=unpavedRoads&key="+key)
    
    if response.status_code != 200:
        raise Exception("Something seems to be wrong with the TomTom API service at the moment. Perhaps try again later.")

    my_dict = response.json()

    for i in my_dict:
        my_lst = my_dict[i]

    new_dict = my_lst[0]

    direction_lst = new_dict["legs"][0]["points"]


    for i in range(len(direction_lst)):
        route_lon.append(float(direction_lst[i]["longitude"]))
        route_lat.append(float(direction_lst[i]["latitude"]))

random_num = random.randint(0,len(customer_dict)-1)
counter = 0

for customer in customer_dict:
    if counter == random_num:
        route_to_customer(customer_dict[customer])
        break
    counter += 1

for i in range(len(route_lat) - 1):
    folium.PolyLine(locations=[(route_lat[i],route_lon[i]),(route_lat[i+1],route_lon[i+1])]).add_to(m)

m